# 7.2 Graph properties — every number is a claim about a rule

[07.1](07.1-social_graphs.ipynb) built the same messages into two networks and stopped at
the picture. This notebook is about the numbers you are allowed to put next to that picture,
and about one fact that runs through all of them: **a centrality score is computed from the
edges, and the edges were your decision.** Change the rule and the score changes, so the
score is a statement about the rule at least as much as about the person it is attached to.

The order is deliberate. First a graph small enough to read by eye, where the edges are
documented and the answer is known from outside the data — that is the calibration, the
place where you find out what degree and betweenness *mean* when they behave. Then the IRC
chat graph, which is four hundred nodes, has no documented edges at all, and where the same
three measures behave rather worse.

Barabási's [*Network Science*](http://networksciencebook.com/) chapter
[2](http://networksciencebook.com/chapter/2) is the reference for everything here: degree and
the degree distribution in 2.3, the adjacency matrix in 2.4, weighted networks in 2.6, paths
and distances in 2.8, connectedness in 2.9, the clustering coefficient in 2.10. It is free
and it is short.

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
from goad_toolkit.visualizer import PlotSettings, ScatterPlot
from scipy.stats import spearmanr

from scripts.graphs import GraphPlot, MentionEdges, giant_component, to_graph
from scripts.pipelines import BuildTimestamp, build_irc_pipeline
from scripts.plots import BarPlot
from wa_analyzer.data import load_showcase
from wa_analyzer.network_analysis import Config, GraphBuilder

## 7.2.1 A graph where the answer is known

Fifteen Florentine families in the early 1400s, an edge for a marriage between them. It is a
famous dataset because the answer is known from the historical record and not from the graph:
the Medici ended up running Florence, and the question is whether the marriage network saw it
coming.

Three measures, all of them one line of `networkx`:

- **Degree** — how many families you married into. $d_u = \sum_{v} A[u,v]$, the row sum of the
  adjacency matrix.
- **Betweenness** — the share of all shortest paths between other pairs that run through you.
  $c_b(v) = \sum_{s,t} \frac{\sigma(s,t \mid v)}{\sigma(s,t)}$, where $\sigma(s,t)$ counts the
  shortest paths from $s$ to $t$ and $\sigma(s,t \mid v)$ counts those that pass through $v$.
- **Clustering coefficient** — of all the pairs of your neighbours, what share are connected to
  each other. High means your contacts already know each other.

In [ ]:
florence = nx.florentine_families_graph()
families = pd.DataFrame(
    {
        "degree": dict(florence.degree()),
        "betweenness": nx.betweenness_centrality(florence),
        "clustering": nx.clustering(florence),
    }
).sort_values("betweenness", ascending=False)
print(families.round(3).to_string())
print(f"\n{florence.number_of_nodes()} families, {florence.number_of_edges()} marriages, "
      f"density {nx.density(florence):.2f}, diameter {nx.diameter(florence)}")

Read the top three rows against each other, because that is where the three measures stop
agreeing.

**The Medici have both the highest degree (6) and the highest betweenness (0.52), and those are
not the same achievement.** Their betweenness is twice the next family's, on a graph where the
most-married family has only six ties: half of every shortest path in Florence runs through
them. Their clustering coefficient is 0.07, which on six neighbours means exactly one of the
fifteen pairs of families they married into had also married each other. Fourteen of those
fifteen connections exist only by going back through the Medici.

**Strozzi and Guadagni both have degree 4.** By the measure a reader is most likely to be shown
they are equals. Their betweenness differs by a factor of two and a half — 0.26 against 0.10 —
because the Guadagni sit between parts of the network that have no other route to each other
(clustering 0.00: none of their four in-laws are connected), and the Strozzi married into a
corner that was already well connected (clustering 0.33). Peruzzi are the extreme case: degree 3,
clustering 0.67, betweenness 0.02. They are inside a clique, and a clique needs no broker.

This is what these measures are for. Degree counts your contacts; betweenness asks whether
anything would break if you left; the clustering coefficient asks whether your contacts need you
in order to reach each other. And the reason we are entitled to trust that reading is that the
answer can be checked against something outside the graph — the historical record of who ended
up governing Florence. Nothing below has that.

In [ ]:
top = families.head(8).reset_index(names="family")
bars = PlotSettings(figsize=(11, 4), title="Same families, three orderings", max_cols=3,
                    subplot_titles=["degree", "betweenness", "clustering coefficient"], ylabel="")
host = BarPlot(bars)
fig, axes = host.create_figure(n_plots=3)
for ax, column in zip(axes, ["degree", "betweenness", "clustering"]):
    colours = ["crimson" if f == "Medici" else "lightgrey" for f in top.sort_values(column, ascending=False).family]
    host.plot_on_axes(BarPlot(bars), ax, data=top.sort_values(column, ascending=False),
                      x=column, y="family", palette=colours, hue="family", legend=False)
    ax.set_ylabel("")

## 7.2.2 The same three measures on a chat

Now the IRC showcase, through 07.1's two rules on the same 2015 messages from `#ubuntu-uk`:
the **nearby** rule (an edge when two people spoke within ten minutes) and the **mention**
rule (an edge when one typed the other's nick).

`MentionEdges` is 07.1's `groupby` with its `isin(nicks)` guard, moved into `scripts/graphs.py`
so no graph gets built without it — a third of what the `addressed_to` regex catches is words
like `yeah,` rather than people.

In [ ]:
pipeline = build_irc_pipeline()
pipeline.add(BuildTimestamp)
irc = pipeline.apply(load_showcase("ubuntu_irc"))
chat = irc[(irc.channel == "#ubuntu-uk") & (irc.date.dt.year == 2015)].copy()

builder = GraphBuilder(Config(time_col="timestamp", node_col="author", seconds=600, datafile=None))
nearby = builder.build(chat, edge_seconds=600)
named = to_graph(MentionEdges()(chat), nodes=chat.author.unique())

for label, graph in [("nearby", nearby), ("named", named)]:
    components = sorted(nx.connected_components(graph), key=len, reverse=True)
    isolated = sum(1 for node in graph.nodes() if graph.degree(node) == 0)
    print(f"{label:7s} {graph.number_of_nodes()} nodes, {graph.number_of_edges():,} edges, "
          f"density {nx.density(graph):.4f}, {len(components)} components "
          f"(largest {len(components[0])}), {isolated} isolated")

**The component count is the first honest number here, and it is not in the picture.** The
mention graph is 188 pieces: one lump of 256 people and 187 nicks that nobody ever addressed
and who never addressed anybody. They are in the channel — they sent messages, they are nodes —
and the rule connects them to nothing. The nearby rule leaves only 39 such people, because
turning up at a busy moment is easier than being named.

Every measure from here on is computed on the largest component, which is the usual thing to do
and which quietly drops 42% of the people in one graph and 9% in the other. That belongs in the
caption too.

In [ ]:
core = giant_component(nearby)
core_named = giant_component(named)
print(f"nearby giant: {core.number_of_nodes()} nodes, average clustering "
      f"{nx.average_clustering(core):.3f}, diameter {nx.diameter(core)}, "
      f"average shortest path {nx.average_shortest_path_length(core):.2f}")
print(f"named giant:  {core_named.number_of_nodes()} nodes, average clustering "
      f"{nx.average_clustering(core_named):.3f}, diameter {nx.diameter(core_named)}, "
      f"average shortest path {nx.average_shortest_path_length(core_named):.2f}")

Both graphs have diameter 5 and an average path just over two: from anyone to anyone in about
two steps. That sounds like a finding about a tight-knit community and it is nothing of the
kind — a graph this dense, built from a rule this generous, could hardly come out otherwise.
The number to be surprised by would have been a *large* diameter.

In [ ]:
scores = pd.DataFrame(
    {
        "degree": dict(core.degree()),
        "weighted_degree": dict(core.degree(weight="weight")),
        "betweenness": nx.betweenness_centrality(core),
        "clustering": nx.clustering(core),
    }
)
scores["messages"] = chat.author.value_counts().reindex(scores.index)
scores["named_degree"] = pd.Series(dict(core_named.degree())).reindex(scores.index).fillna(0)
print("top 10 by degree under the nearby rule:")
print(scores.nlargest(10, "degree").round(3).to_string())

## 7.2.3 The bot problem is a rule problem

`lubotu3\`` is the channel's bug-tracker bot: it posts a link whenever somebody types a bug
number. It sends very few messages and it is present at every busy moment, which is exactly
what the nearby rule rewards.

In [ ]:
messages = chat.author.value_counts()
rank = scores.degree.rank(ascending=False, method="min")
peers = messages[(messages >= 25) & (messages <= 50)].index
peer_degree = scores.degree.reindex(peers).dropna()
reach = scores[scores.degree >= scores.degree.get("lubotu3`", 0)].messages

print(f"lubotu3`: {messages['lubotu3`']} messages, degree {scores.degree['lubotu3`']:.0f}, "
      f"ranked {rank['lubotu3`']:.0f} of {len(scores)} for degree")
print(f"humans who sent 25-50 messages: median degree {peer_degree.median():.0f}, "
      f"highest {peer_degree.max():.0f}")
print(f"everyone else at that degree or above sent a median of {reach.median():,.0f} messages")

**Thirty-six messages buys the thirty-sixth-highest degree of the 404 accounts the rule connects
to anybody.** The other accounts up at that level sent a median of 1,350 messages each. Nobody
ever addressed the bot by name, so under the mention rule it has degree 1 and sits near the
bottom — 07.1 already showed that pair of numbers.

The useful framing is not "drop the bots". It is that *the nearby rule measures presence*, and
a script that is always present scores highly on it because that is what it was asked to
measure. A bot is only the most obvious account for which "was in the room" and "took part" come
apart; a person who leaves an IRC client open all day is the same case with a less obvious name.
You will not find those by grepping for `bot`.

## 7.2.4 Weighted degree, and a correlation worth checking

`GraphBuilder` stores a weight on every edge: how many times the pair fell inside the same
window. Summing those is *weighted degree*, and it is the natural next measure to reach for
(Barabási 2.6). Before reading anything into it, check it against the column you already had.

In [ ]:
paired = scores.dropna(subset=["messages"])
for column in ["degree", "weighted_degree", "betweenness", "clustering"]:
    rho = spearmanr(paired[column], paired.messages).statistic
    print(f"spearman({column:16s}, messages sent) = {rho:+.2f}")
print(f"\nspearman(betweenness, degree)          = "
      f"{spearmanr(paired.betweenness, paired.degree).statistic:+.2f}")

**Weighted degree correlates 0.92 with the message count.** It is the number of messages, run
through a graph and back out again. If a report leads with "X has the highest weighted degree in
the network", a reader who has the message counts already knows that, and the graph has added a
layer of machinery and no information. Say the message count; it is the honest version of the
same sentence and nobody has to trust your edge rule to believe it.

Betweenness is barely better here. It correlates 0.73 with messages sent and 0.80 with degree, so
whatever it is picking up is mostly "this person is busy". Compare that with the Florentine graph,
where betweenness ranked Guadagni above Strozzi at equal degree and the ranking meant something.
The measure did not change. The graph did: the nearby rule connects everyone who was awake at the
same time, so there are no structural holes left for a broker to sit in.

**The check costs one line and it is not optional.** A centrality that reproduces a column you
already have is not a finding, and it is very easy to present as one.

In [ ]:
settings = PlotSettings(
    figsize=(7, 6),
    title="Weighted degree recovers the message count (spearman 0.92)",
    xlabel="messages sent in 2015",
    ylabel="weighted degree, nearby rule",
)
fig, ax = ScatterPlot(settings).plot(data=paired, x="messages", y="weighted_degree", color="#cccccc")
loud = paired.nlargest(4, "messages")
ax.scatter(loud.messages, loud.weighted_degree, color="#c44e52", zorder=3)
for row in loud.itertuples():
    ax.annotate(f"  {row.Index}", (row.messages, row.weighted_degree), color="#c44e52", va="center")
ax.set_xscale("log")
ax.set_yscale("log")

## 7.2.5 Two rules, two clustering coefficients, opposite signs

The clustering coefficient asks whether your neighbours know each other. Run it under both edge
rules on the same people and correlate it with degree.

In [ ]:
shared = sorted(set(core.nodes()) & set(core_named.nodes()))
both = pd.DataFrame(
    {
        "nearby_clustering": pd.Series(nx.clustering(core)).reindex(shared),
        "nearby_degree": pd.Series(dict(core.degree())).reindex(shared),
        "named_clustering": pd.Series(nx.clustering(core_named)).reindex(shared),
        "named_degree": pd.Series(dict(core_named.degree())).reindex(shared),
    }
)
print(f"{len(both)} people in both giant components")
print(f"average clustering, nearby rule: {both.nearby_clustering.mean():.3f}")
print(f"average clustering, mention rule: {both.named_clustering.mean():.3f}")
print(f"spearman(clustering, degree) under the nearby rule:  "
      f"{spearmanr(both.nearby_clustering, both.nearby_degree).statistic:+.2f}")
print(f"spearman(clustering, degree) under the mention rule: "
      f"{spearmanr(both.named_clustering, both.named_degree).statistic:+.2f}")

Same people, same year, same channel, and the sign flips. Under the nearby rule the busiest
accounts have the *lowest* clustering — they are around at every hour, so their neighbourhood
includes the morning crowd and the late-night crowd, who never overlap with each other. Under
the mention rule the busiest accounts have the *highest* clustering, because a conversation you
have to be named in is a conversation several people are in together.

Neither sign is wrong. "Hubs sit between otherwise separate groups" and "hubs sit inside dense
groups" are both defensible sentences about `#ubuntu-uk` in 2015, and which one you get is
decided by a line of code that nobody reading your report will see. **This is the whole point of
the lesson.** If you write one of those sentences, write the edge rule next to it.

## 7.2.6 Why you cannot compare a centrality across graphs

Betweenness in `networkx` is normalised by default: divided by the number of pairs it could
possibly sit between, so it lands in $[0, 1]$ and looks comparable. It is not.

In [ ]:
rows = []
for label, graph in [("florentine", florence), ("karate club", nx.karate_club_graph()),
                     ("irc nearby giant", core)]:
    normalised = nx.betweenness_centrality(graph)
    raw = nx.betweenness_centrality(graph, normalized=False)
    rows.append({
        "graph": label, "nodes": graph.number_of_nodes(),
        "max normalised": max(normalised.values()), "mean normalised": np.mean(list(normalised.values())),
        "max raw": max(raw.values()), "max degree": max(dict(graph.degree()).values()),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))

The most central family in a fifteen-node network scores 0.52. The most central account in a
four-hundred-node network scores 0.18. On the raw scale the ranking reverses and the ratio is
three hundred to one. Both columns are correct arithmetic and neither supports the sentence
"the Medici were more central to Florence than daftykins was to `#ubuntu-uk`".

The reason is that normalising divides by the number of *pairs*, which grows with $n^2$, while
the paths a single node can actually capture do not. Maximum degree has the same problem in the
other direction: 6 against 215, and the second graph is not thirty-five times more social. **A
centrality is a ranking within one graph. Across graphs it is not a quantity.** If you need to
compare two networks, compare their rankings — is the top account also top under the other
rule — or compare each against a graph of the same size, which is what
[07.3](07.3-clusters-in-graphs.ipynb) does for modularity.

## 7.2.7 Position in a layout is not data

Degree, betweenness and clustering are properties of the graph. The picture is not: a spring
layout treats edges as springs and nodes as repelling charges and settles wherever it settles,
which depends on the seed. Two layouts of one graph, coloured by the same degrees.

In [ ]:
hubs = giant_component(to_graph(MentionEdges()(chat)))
hubs = hubs.subgraph([n for n in hubs.nodes() if hubs.degree(n) >= 6])
hubs = giant_component(nx.Graph(hubs))
degrees = np.array([hubs.degree(n) for n in hubs.nodes()], dtype=float)

layout = PlotSettings(figsize=(11, 5), max_cols=2, title="One graph, two seeds, identical degrees",
                      subplot_titles=["spring layout, seed 1", "spring layout, seed 7"])
host = GraphPlot(layout)
fig, axes = host.create_figure(n_plots=2)
for ax, seed in zip(axes, [1, 7]):
    host.plot_on_axes(GraphPlot(layout), ax, data=hubs,
                      pos=nx.spring_layout(hubs, seed=seed, k=0.35),
                      color=degrees, node_size=3 * degrees, cmap="viridis")
print(f"{hubs.number_of_nodes()} people addressed by at least six others; "
      f"highest degree {int(degrees.max())}, median {np.median(degrees):.0f}")

The two panels look like different networks. They are the same one, and every degree, every
betweenness and every clustering coefficient is identical between them. So: **never write that
someone is "on the edge of the network"** when what happened is that this layout put them there.
Write their degree, which will still be true tomorrow with a different seed.

The thing that *is* in the picture is the degree, because we mapped it to colour and size on
purpose. That mapping is the caption's job.

## 7.2.8 The long tail arrives here too

One last number, and it is the reason a summary statistic of this graph describes nobody. Lesson
4's question — is this one population or several — applies to a degree distribution as much as to
a flipper length (Barabási 2.3, and chapter [4](http://networksciencebook.com/chapter/4) for what
a heavy tail does to a network).

In [ ]:
all_degrees = pd.Series(dict(nearby.degree()))
touched = {node for node, _ in all_degrees.nlargest(10).items()}
carried = sum(1 for u, v in nearby.edges() if u in touched or v in touched)
print(all_degrees.describe().round(1).to_string())
print(f"\nshare of accounts with degree <= 5: {(all_degrees <= 5).mean():.1%}")
print(f"share of edges touching the ten highest-degree accounts: "
      f"{carried:,}/{nearby.number_of_edges():,} = {carried / nearby.number_of_edges():.1%}")

Mean degree 14.7, median 5, maximum 215. Over half the channel has five neighbours or fewer, and
ten accounts out of 443 sit on nearly half the edges. "The average person in this network has
fifteen connections" is arithmetically true and describes nobody: not the 55% below five, and
certainly not the ten who hold the thing together.

## 7.2.9 Your turn

Your own chat is smaller and denser than an IRC channel — a group of eight is close to complete
at any window, so degree will separate almost nobody. That is the point of doing it: the measure
does not fail loudly, it just returns numbers that all look alike, and you have to notice.

```python
from scripts.graphs import giant_component
from wa_analyzer.data import load_own_chat
from wa_analyzer.network_analysis import Config, GraphBuilder

own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])
graph = GraphBuilder(Config(time_col="timestamp", node_col="author",
                            seconds=600, datafile=None)).build(own, edge_seconds=600)

scores = pd.DataFrame({
    "degree": dict(graph.degree()),
    "weighted_degree": dict(graph.degree(weight="weight")),
    "betweenness": nx.betweenness_centrality(giant_component(graph)),
    "clustering": nx.clustering(giant_component(graph)),
})
scores["messages"] = own.author.value_counts().reindex(scores.index)
print(spearmanr(scores.weighted_degree, scores.messages).statistic)
```

> **Your turn.**
>
> 1. Run the correlation between weighted degree and message count on your own chat, as §7.2.4
>    did. If it is above 0.9, say so — you have found that your centrality is a message counter,
>    which is a result and saves you from writing a sentence you cannot defend.
> 2. Compute the clustering coefficient at a 60-second window and at a 30-minute window. Report
>    both. If the ordering of people changes between them, the ordering is a property of the
>    window.
> 3. Count the components and the isolated nodes. In a group chat there will be one component and
>    no isolates, which tells you that the *rule* cannot separate anybody in a chat this size —
>    not that everybody is equally close.
> 4. Find your `lubotu3\``: an account whose degree is high relative to how much it says. In a
>    WhatsApp export that is often the person who only ever reacts, or a number with no name.

## 7.2.10 What to write down

1. **The edge rule, again**, and this time next to the measure. Not "degree" but "degree under a
   ten-minute window".
2. **Which component you measured on**, and how many people that dropped.
3. **One correlation between your centrality and a column you already had.** If it is high, that
   is your finding.
4. **A number, not a layout.** If your evidence for someone being central is where the spring
   layout put them, you have no evidence.
5. **Nothing compared across two graphs of different size** unless you compared rankings.

---

**Where this goes next.** Every measure in this notebook is about one node at a time. The next
question is about groups: does this network split into communities, and would you know if the
split were an artefact? [07.3-clusters-in-graphs](07.3-clusters-in-graphs.ipynb) does it on the
karate club, where the true split is recorded, and then on this chat graph, where it is not.